In [24]:
import pickle as pkl
import random
import numpy as np
import os
import sys
from pathlib import Path
import pandas as pd

NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name != "notebooks":
    if (NOTEBOOKS_DIR / "notebooks").exists():
        NOTEBOOKS_DIR = NOTEBOOKS_DIR / "notebooks"
    elif NOTEBOOKS_DIR.parent.name == "notebooks":
        NOTEBOOKS_DIR = NOTEBOOKS_DIR.parent
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

from config import PATHS

# GO analysis
from goatools.obo_parser import GODag
from goatools.associations import read_gaf

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Control flag for embedding type
USE_GF = True

In [25]:
IDH1_PATH = PATHS["idh1_prkdc_inference_dir"] + '/'

PRMT5_PATH = PATHS["prmt5_mat2a_inference_dir"] + '/'
idh1_res = os.listdir(IDH1_PATH)
prmt5_res = os.listdir(PRMT5_PATH)

In [26]:
idh1_scoring_matrix = np.zeros((len(idh1_res), 8)) # 8 cancers
for idx in range(len(idh1_res)):
    df = pd.read_csv(IDH1_PATH + idh1_res[idx])['avg_score']
    idh1_scoring_matrix[idx] = df.values
idh1_scoring_matrix.shape

(70, 8)

In [27]:
prmt5_scoring_matrix = np.zeros((len(prmt5_res), 7)) # 7 cancers
for idx in range(len(prmt5_res)):
    df = pd.read_csv(PRMT5_PATH + prmt5_res[idx])['avg_score']
    prmt5_scoring_matrix[idx] = df.values
prmt5_scoring_matrix.shape

(70, 7)

In [28]:
idh1_glioma = idh1_scoring_matrix[:, -1]
print(idh1_glioma)
print(np.argmax(idh1_glioma))
print(idh1_glioma[np.argmax(idh1_glioma)])
idh1_glioma_mean = np.mean(idh1_glioma.reshape(-1, 1), axis=0)
idh1_glioma_mean

[0.27337438 0.20391949 0.15090322 0.21191049 0.21416478 0.10835943
 0.10073197 0.20902753 0.20907219 0.39980268 0.2596621  0.13761726
 0.3719191  0.32927644 0.44465336 0.36366978 0.20586474 0.3051516
 0.15777864 0.17355087 0.15939042 0.1789848  0.34447303 0.20096543
 0.1005797  0.31148297 0.32306886 0.33107054 0.18298395 0.4940067
 0.4039505  0.25182098 0.15515463 0.2080874  0.5336286  0.30455193
 0.13861957 0.22318728 0.14053762 0.33787817 0.2490693  0.3004572
 0.2522408  0.22205468 0.12958615 0.17003375 0.15020294 0.20383433
 0.18883196 0.06894847 0.24010694 0.33457676 0.2055037  0.29667675
 0.1932096  0.25453883 0.2536336  0.20869689 0.14094712 0.36234003
 0.40188065 0.18325077 0.1575454  0.2154098  0.40619126 0.19585904
 0.38752145 0.37516633 0.13655064 0.28815016]
34
0.5336286


array([0.24754069])

In [29]:
prmt5_glioma = prmt5_scoring_matrix[:, -1]
prmt5_glioma = np.mean(prmt5_glioma.reshape(-1, 1), axis=0)
prmt5_glioma

array([0.14186052])

In [30]:
with open(PATHS["gene2id_map"], 'rb') as f: 
    gene2id_map = pkl.load(f)
    
with open(PATHS["cancer_list"]) as f:
    cancer_list = [line.rstrip('\n') for line in f]

# Create reverse mappings
id2gene_map = {i:g for g,i in gene2id_map.items()}
id2cancer_map = {i:cancer for i,cancer in enumerate(cancer_list)}
cancer2id_map = {cancer:i for i,cancer in enumerate(cancer_list)}

print(f"Loaded {len(gene2id_map)} genes and {len(cancer_list)} cancer types")
print(f"Cancer types: {cancer_list}")

Loaded 13459 genes and 9 cancer types
Cancer types: ['KIRC', 'COAD', 'LAML', 'OV', 'BRCA', 'CESC', 'SKCM', 'LUAD', 'Glioma']


In [31]:
go_dag = GODag(PATHS["go_basic_obo"])
annotations = read_gaf(PATHS["goa_human_gaf"], go_dag=go_dag)

# Load ID mapping from UniProt to gene symbols
id_mapping = pd.read_csv(PATHS["idmapping_tsv"], sep='\t')
id_mapping = dict(zip(id_mapping['From'], id_mapping['To']))

# Map annotations to gene symbols
anno_mapped = {}
for k, v in annotations.items():
    if k in id_mapping:
        anno_mapped[id_mapping[k]] = v

print(f"Loaded GO annotations for {len(anno_mapped)} genes")
print(f"Total GO terms in ontology: {len(go_dag)}")

/home/jienihu/sc/SLformer/data/GO/go-basic.obo: fmt(1.2) rel(2024-06-17) 45,494 Terms
HMS:0:00:16.948835 707,170 annotations READ: /home/jienihu/sc/SLformer/data/GO/goa_human.gaf 
36129 IDs in loaded association branch, BP
Loaded GO annotations for 17575 genes
Total GO terms in ontology: 45494


In [32]:
def get_go_term_details(go_terms):
    """Get detailed information about GO terms."""
    details = []
    for term_id in go_terms:
        if term_id in go_dag:
            term = go_dag[term_id]
            details.append({
                'id': term_id,
                'name': term.name,
                'namespace': term.namespace,
                'depth': term.depth
            })
    return details

def find_shared_goterms(gene1, gene2, return_terms=False):
    """Find shared GO terms between two genes."""
    go_terms_gene1 = {go_id for go_id in anno_mapped.get(gene1, [])}
    go_terms_gene2 = {go_id for go_id in anno_mapped.get(gene2, [])}
    
    shared_go_terms = go_terms_gene1.intersection(go_terms_gene2)
    gene1_terms = [terms['name'] for terms in get_go_term_details(go_terms_gene1)]
    gene2_terms = [terms['name'] for terms in get_go_term_details(go_terms_gene2)]
    shared_go_terms = [terms['name'] for terms in get_go_term_details(shared_go_terms)]

    if return_terms:
        return len(shared_go_terms), shared_go_terms, gene1_terms, gene2_terms
    else:
        return len(shared_go_terms)
    
def get_gene_sentence_neighbors(gene, cancer, common_data, top_n=10):
    """Get top co-expressed neighbors from gene sentence."""
    gene2id_map = common_data["gene2id_map"]
    gene2sent_map = common_data["gene2sent_map"]
    cancer2id_map = common_data["cancer2id_map"]
    
    if gene not in gene2id_map:
        return []
    
    gene_id = gene2id_map[gene]
    cancer_id = cancer2id_map[cancer]
    
    if cancer_id not in gene2sent_map or gene_id not in gene2sent_map[cancer_id]:
        return []
    
    # Get gene sentence (list of transformed gene IDs)
    gene_sentence = gene2sent_map[cancer_id][gene_id]
    
    # Convert back to gene names
    id2gene_map = {i:g for g,i in gene2id_map.items()}
    neighbors = []
    ngene = len(gene2id_map)  # Total number of genes
    
    for gene_idx in gene_sentence[1:top_n+1]:  # Skip first gene (itself)
        if gene_idx > 0:  # Skip padding tokens
            # Reverse the transformation: (g+1)+cancer*(ngene+1)
            # So: original_gene_id = (gene_idx - cancer_id * (ngene + 1)) - 1
            original_gene_id = (gene_idx - cancer_id * (ngene + 1)) - 1
            
            if original_gene_id in id2gene_map:
                neighbors.append(id2gene_map[original_gene_id])
            else:
                print(f"  Gene ID {gene_idx} -> {original_gene_id} not found in id2gene_map")
    
    return neighbors


In [33]:
gene_preds = pd.DataFrame()
for i in range(5):
    path = f'data/all_SL/pred_mix_slformer_kg_cv{i+1}.csv'
    df = pd.read_csv(path)
    gene_preds = pd.concat([gene_preds, df], ignore_index=True)

def search_pred_score(gene_pair, context=None):
    if context:
        df = gene_preds[((gene_preds['primary_gene'] == gene_pair[0]) & (gene_preds['partner_gene'] == gene_pair[1]) | (gene_preds['primary_gene'] == gene_pair[1]) & (gene_preds['partner_gene'] == gene_pair[0])) & (gene_preds['cancer'] == context)]
    else:
        df = gene_preds[((gene_preds['primary_gene'] == gene_pair[0]) & (gene_preds['partner_gene'] == gene_pair[1]) | (gene_preds['primary_gene'] == gene_pair[1]) & (gene_preds['partner_gene'] == gene_pair[0]))]
    if df.empty:
        print("end= No prediction found for the gene pair:", gene_pair)
        return 
    return df['score'].to_list() if not context else df['score'].to_list()[0]

In [34]:
IDH1_EMB_PATH = PATHS["idh1_prkdc_emb_dir"]
BEST_FOLD = 3
idh1_embs = pd.DataFrame()
scoring_df = pd.read_csv(IDH1_PATH + idh1_res[np.argmax(idh1_glioma)])
score = scoring_df[f'score_{BEST_FOLD}']


contexts_avail = os.listdir(IDH1_EMB_PATH)
for context in contexts_avail:
    with_context_path = IDH1_EMB_PATH + f'/{context}'
    emb_file = os.listdir(with_context_path)[2 * np.argmax(idh1_glioma)]
    emb = pkl.load(open(with_context_path + f'/{emb_file}', 'rb'))
    idh1_emb, prkdc_emb = emb[BEST_FOLD-1][0], emb[BEST_FOLD-1][1]
    pair_emb = np.concatenate([idh1_emb, prkdc_emb], axis=0)
    idh1_embs = pd.concat([idh1_embs, pd.DataFrame(pair_emb).T], ignore_index=True)
    
idh1_embs['context'] = contexts_avail
idh1_embs['SLscore'] = score
idh1_embs.columns = [f'primary_emb_{i}' for i in range(idh1_emb.shape[0])] + [f'partner_emb_{i}' for i in range(prkdc_emb.shape[0])] + ['context', 'SLscore']
idh1_embs

,primary_emb_0,primary_emb_1,primary_emb_2,primary_emb_3,primary_emb_4,primary_emb_5,primary_emb_6,primary_emb_7,primary_emb_8,primary_emb_9,...,partner_emb_504,partner_emb_505,partner_emb_506,partner_emb_507,partner_emb_508,partner_emb_509,partner_emb_510,partner_emb_511,context,SLscore
0,0.165571,0.000000,0.097131,0.152494,0.090950,0.096055,0.161412,0.205624,0.089460,0.098780,...,0.057604,0.012493,0.251751,0.384084,0.000000,0.000000,0.103857,0.347548,COAD,0.012469
1,0.025391,0.000000,0.000000,0.020430,0.078672,0.198429,0.412788,0.314775,0.030255,0.000092,...,0.281317,0.003405,0.178923,0.277670,0.049380,0.118358,0.000000,0.384764,LAML,0.084339
2,0.094682,0.013012,0.284993,0.004009,0.077009,0.076318,0.352992,0.038492,0.380259,0.059880,...,0.052812,0.000000,0.393915,0.259869,0.102362,0.123124,0.005940,0.775608,OV,0.268113
3,0.000000,0.000000,0.149566,0.245552,0.029750,0.133357,0.089524,0.128748,0.026488,0.244730,...,0.198616,0.001938,0.365141,0.473631,0.454249,0.004700,0.091806,0.232266,BRCA,0.435550
4,0.191499,0.003051,0.072826,0.080107,0.210498,0.242653,0.283781,0.081931,0.128906,0.009789,...,0.151785,0.006049,0.341714,0.067987,0.007269,0.150766,0.007905,0.631405,CESC,0.002941
5,0.026961,0.000000,0.000000,0.002221,0.001730,0.041531,0.074549,0.118929,0.117865,0.503274,...,0.222365,0.000000,0.254017,0.011614,0.189765,0.226391,0.003208,0.043716,SKCM,0.020995
6,0.055962,0.000000,0.067023,0.068055,0.067863,0.220504,0.083298,0.086192,0.022402,0.202265,...,0.235961,0.004871,0.263633,0.023978,0.198554,0.004168,0.000272,0.258095,LUAD,0.417944
7,0.059640,0.000000,0.226064,0.043419,0.002492,0.277073,0.197727,0.111201,0.079738,0.129580,...,0.083635,0.000000,0.263663,0.368031,0.303156,0.009088,0.002110,0.005268,Glioma,0.903764


In [35]:
# gene_pair = ['PARP1', 'BRCA1']
# gene_pair_map = {
#     'primary': 'BRCA1',
#     'partner': 'PARP1'
# }
# context = 'BRCA'
# score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['KRAS', 'TBK1']
# gene_pair_map = {
#     'primary': 'KRAS',
#     'partner': 'TBK1'
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)

# ===============================================================

gene_pair = ['IDH1', 'PRKDC']
gene_pair_map = {
    'primary': 'IDH1',
    'partner': 'PRKDC'
}
context = 'Glioma'
score = idh1_embs['SLscore']

# ===============================================================

# gene_pair = ['PRMT5', 'MAT2A']
# gene_pair_map = {
#     'primary': 'MAT2A',
#     'partner': 'PRMT5'
# }
# context = 'Glioma'
# score = prmt5_glioma[0]

# ===============================================================

# gene_pair = ['BRCA2', 'EZH2']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'BRCA'
# score = search_pred_score(gene_pair, context=context)

# ===============================================================

# gene_pair = ['ERH', 'KRAS']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['GATA2', 'KRAS']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)
# ===============================================================

# gene_pair = ['KRAS', 'PLK1']
# gene_pair_map = {
#     'primary': gene_pair[0],
#     'partner': gene_pair[1]
# }
# context = 'LUAD'
# score = search_pred_score(gene_pair, context=context)

In [36]:
print(f"Current embedding type: {'Geneformer' if USE_GF else 'Cross-attention'}")

Current embedding type: Geneformer


In [37]:
def get_common_data(sent_n=200):

    common_data_path = {
        'geneformer_emb_map': PATHS["geneformer_emb_map"], 
        'geneformer_emb_mtx': PATHS["geneformer_emb_mtx"], 
        "gene2sent_map": PATHS["gene2sent_map_template"].format(sent_n=sent_n),
        "sent_mask_map": PATHS["sent_mask_map_template"].format(sent_n=sent_n),
        "gene2id_map": PATHS["gene2id_map"],
    }
    
    # print(common_data_path)
    for data, path in common_data_path.items():
        if not os.path.exists(path):
            raise Exception(f"{data,path} cannot be found, please first construct it.")
    
    common_data = {}
    for data in ["geneformer_emb_map","gene2sent_map","sent_mask_map","gene2id_map"]:
        with open(common_data_path[data], 'rb') as f:
            common_data[data] = pkl.load(f)
    common_data["geneformer_emb_mtx"] = np.load(common_data_path["geneformer_emb_mtx"])
     
    cancer2id_map = {c:i for i,c in enumerate(cancer_list)}
    common_data["cancer_list"] = cancer_list
    common_data["cancer2id_map"] = cancer2id_map
    go_anno = pd.read_csv(PATHS["go_anno_popular_csv"])
    gene2go_map = dict(zip(go_anno["gene_id"], go_anno["GO_id"]))
    # gene2go_map = dict(zip(go_anno["gene_id"], go_anno["annotation"]))
    common_data["gene2go_map"] = gene2go_map
    
    # Load co-expression data
    common_data["cos_sim_kg"] = pd.read_csv(PATHS["cos_sim_kg_csv"])
    common_data["func_cos_sim_kg"] = pd.read_csv(PATHS["func_cos_sim_kg_csv"])
    
    return common_data
common_data = get_common_data()

In [38]:
def load_embedding_features(gene_pair_map, embedding_base_path="embedding_saved", use_gf=None):
    """Load important embedding features for a gene pair across all contexts."""
    if use_gf is None:
        use_gf = USE_GF  # Use global flag if not specified
    
    primary_gene = gene_pair_map['primary']
    partner_gene = gene_pair_map['partner']
    
    # Choose embedding type directory based on USE_GF flag
    embedding_type_dir = "geneformer_important" if use_gf else "crossemb_important"
    
    # Construct the directory path for this gene pair
    gene_pair_dir = f"{primary_gene}-{partner_gene}"
    full_path = os.path.join(embedding_base_path, embedding_type_dir, gene_pair_dir)
    
    if not os.path.exists(full_path):
        # Try reverse order
        gene_pair_dir = f"{partner_gene}-{primary_gene}"
        full_path = os.path.join(embedding_base_path, embedding_type_dir, gene_pair_dir)
        print(full_path)
        if not os.path.exists(full_path):
            print(f"Warning: No embedding features found for {primary_gene}-{partner_gene} using {'Geneformer' if use_gf else 'cross-attention'} embeddings")
            return None, None
    
    # Load primary and partner embedding features
    primary_features_path = os.path.join(full_path, "primary_important_features.csv")
    partner_features_path = os.path.join(full_path, "partner_important_features.csv")
    
    primary_features = None
    partner_features = None
    
    primary_features = pd.read_csv(primary_features_path)
    partner_features = pd.read_csv(partner_features_path)

    
    return primary_features, partner_features

def format_embedding_features(features_df, gene_type="primary"):
    """Format embedding features for prompt inclusion."""
    if features_df is None:
        return f"No {gene_type} embedding features available."
    
    context_data = []
    for _, row in features_df.iterrows():
        cancer = row['cancer']
        score = row['score']
        
        # Get non-zero embedding features
        emb_cols = [col for col in features_df.columns if col.startswith(f'{gene_type}_emb_')]
        active_features = []
        for col in emb_cols:
            if row[col] > 0:
                feature_idx = col.split('_')[-1]
                active_features.append(f"emb_{feature_idx}: {row[col]:.3f}")
        
        context_info = f"  - {cancer} (SL-Score: {score:.3f}): {', '.join(active_features) if active_features else 'No active features'}"
        context_data.append(context_info)
    
    return f"{gene_type.capitalize()} Gene Embedding Features:\n" + "\n".join(context_data)

def _analyze_embedding_contexts(features, gene_type):
    """Helper function to generate per-gene embedding analysis with cross-context comparison."""
    analysis = []
    cancer_types = features['cancer'].unique()
    
    for cancer in cancer_types:
        subset = features[features['cancer'] == cancer]
        score = subset['score'].values[0]
        
        # Analyze activation patterns for this context
        emb_cols = [col for col in features.columns if col.startswith(f'{gene_type}_emb_')]
        row = subset.iloc[0]
        
        # Identify top dimensions and their activation levels
        top_dims = sorted([(col, row[col]) for col in emb_cols], key=lambda x: -x[1])[:5]
        top_dims_str = ', '.join([f"dim_{col.split('_')[-1]}({val:.3f})" for col, val in top_dims])
        
        # Compare to average activation across all contexts
        avg_activations = features[emb_cols].mean().sort_values(ascending=False)
        context_specific = []
        for col, val in top_dims:
            dim_idx = col.split('_')[-1]
            avg_val = avg_activations[col]
            if val > 2*avg_val:  # Arbitrarily defined as context-specific
                context_specific.append(f"dim_{dim_idx}(x{val/avg_val:.1f} vs avg)")
        
        context_specific_str = f"Context-specific: {', '.join(context_specific)}" if context_specific else "No strongly context-specific dimensions"
        
        analysis.append(f"  {cancer} (SL-Score: {score:.3f}): Top dimensions - {top_dims_str} | {context_specific_str}") if not USE_GF else \
        analysis.append(f"  {cancer}: Top dimensions - {top_dims_str} | {context_specific_str}")

    
    return analysis

In [39]:
USE_GF = False

_pair_genes = [gene_pair_map.get('primary'), gene_pair_map.get('partner')]
import re
import gseapy as gp
def _clean_enrichment_terms(terms):
    """Lightly normalize enrichment term strings for better semantic matching.
    - remove species fragments
    - replace underscores
    - strip parentheses-only identifiers
    - lowercase for stability
    """
    cleaned = []
    pat_species = re.compile(r"\s*-?\s*Homo sapiens\b|\s*\(Homo sapiens\)\s*", re.IGNORECASE)
    pat_go_id = re.compile(r"\s*\(GO:\d+\)\s*", re.IGNORECASE)
    for t in terms:
        s = str(t)
        s = pat_species.sub("", s)
        s = pat_go_id.sub("", s)
        s = s.replace("_", " ")
        s = re.sub(r"\s+", " ", s).strip()
        cleaned.append(s.lower())
    return cleaned

def compute_enrichment_text_for_genes(genes, gene_sets=("Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2021"), top_k=10):
    """Query multiple libraries with Enrichr for the provided genes and build a concise text.
    Returns a comma-joined string of the top terms (deduplicated, cleaned) across libraries.
    """
    if not genes:
        return ""
    agg_terms = []
    seen = set()
    # Try each library; keep up to top_k unique terms total
    for gs in gene_sets:
        try:
            enr = gp.enrichr(
                gene_list=list(genes),
                gene_sets=[gs],
                organism='Human',
                outdir=None,
                cutoff=1.0,
            )
            df = enr.results
            if df is None or df.empty:
                continue
            sort_col = "Adjusted P-value" if "Adjusted P-value" in df.columns else ("P-value" if "P-value" in df.columns else None)
            if sort_col is not None:
                df = df.sort_values(sort_col, ascending=True)
            raw_terms = df["Term"].astype(str).tolist()
            for term in _clean_enrichment_terms(raw_terms):
                if term not in seen:
                    agg_terms.append(term)
                    seen.add(term)
                if len(agg_terms) >= top_k:
                    break
            if len(agg_terms) >= top_k:
                break
        except Exception:
            continue
    return ", ".join(agg_terms)

enrichment_text = compute_enrichment_text_for_genes(
    _pair_genes,
    gene_sets=("Reactome_2022", "KEGG_2021_Human", "GO_Biological_Process_2021",
               "GO_Molecular_Function_2021", "GO_Cellular_Component_2021"),
    top_k=20,
)

In [40]:
enrichment_text

'irf3-mediated induction of type i ifn r-hsa-3270619, sting mediated induction of host immune responses r-hsa-1834941, metabolism of cofactors r-hsa-8978934, innate immune system r-hsa-168249, nonhomologous end-joining (nhej) r-hsa-5693571, e3 ubiquitin ligases ubiquitinate target proteins r-hsa-8866654, cytosolic sensors of pathogen-associated dna r-hsa-1834949, peroxisomal protein import r-hsa-9033241, protein ubiquitination r-hsa-8852135, nuclear events mediated by nfe2l2 r-hsa-9759194, immune system r-hsa-168256, keap1-nfe2l2 pathway r-hsa-9755511, dna double-strand break repair r-hsa-5693532, protein localization r-hsa-9609507, cellular response to chemical stress r-hsa-9711123, metabolism of vitamins and cofactors r-hsa-196854, diseases of metabolism r-hsa-5668914, dna repair r-hsa-73894, neutrophil degranulation r-hsa-6798695, cellular responses to stress r-hsa-2262752'

In [41]:
# Soft, data-driven topic nudging toward metabolism and redox cofactors
from typing import Tuple, List

METABOLIC_KEYWORDS = [
    # general
    "metabol", "metabolic process", "biosynthesis", "anabolic", "catabolic",
    # carbon/energy
    "glycolysis", "glycolytic", "gluconeogenesis", "tricarboxylic", "tca", "krebs",
    "oxidative phosphorylation", "oxphos", "electron transport", "warburg",
    "pentose phosphate", "ppp",
    # lipids
    "beta oxidation", "fatty acid", "lipid", "cholesterol", "sterol", "isoprenoid", "mevalonate",
    # one-carbon / nucleotides
    "one-carbon", "folate", "methionine", "s-adenosylmethionine", "sam",
    "nucleotide", "pyrimidine", "purine", "de novo",
    # amino acids / redox
    "amino acid", "glutamine", "glutaminolysis", "glutathione", "redox", "reactive oxygen", "ros",
    # cofactors (general mentions counted under metabolism as well)
    "nadph", "nadp", "nadp+", "nadh", "nad+", "flavin", "fmn", "fad"
]

REDOX_KEYWORDS = [
    # cofactor balance and producers/consumers
    "nadph", "nadp", "nadp+", "nad+/nadh", "nadh", "nad+",
    "pentose phosphate", "g6pd", "glucose-6-phosphate dehydrogenase",
    "6-phosphogluconate dehydrogenase", "pgd",
    "malic enzyme", "me1", "me2",
    "isocitrate dehydrogenase", "idh1", "idh2",
    "thioredoxin", "glutathione", "gpx", "gr", "txnr",
    "ferroptosis", "lipid peroxidation", "oxidative stress",
]

def estimate_metabolic_support(enrichment_text: str) -> Tuple[float, List[str]]:
    """Estimate metabolic-topic support from a comma-joined enrichment summary.
    Returns (support_ratio, representative_terms).
    """
    if not enrichment_text:
        return 0.0, []
    terms = [t.strip().lower() for t in enrichment_text.split(',') if t.strip()]
    if not terms:
        return 0.0, []
    matched = []
    for t in terms:
        if any(kw in t for kw in METABOLIC_KEYWORDS):
            matched.append(t)
    support = len(matched) / max(len(terms), 1)
    # return up to 5 representative matched terms in original order
    seen = set()
    reps = []
    for t in matched:
        if t not in seen:
            reps.append(t)
            seen.add(t)
        if len(reps) >= 5:
            break
    return support, reps


def estimate_redox_support(enrichment_text: str) -> Tuple[float, List[str]]:
    """Estimate redox/NAD(P)H cofactor-related support from enrichment terms.
    Returns (support_ratio, representative_terms).
    """
    if not enrichment_text:
        return 0.0, []
    terms = [t.strip().lower() for t in enrichment_text.split(',') if t.strip()]
    if not terms:
        return 0.0, []
    matched = []
    for t in terms:
        if any(kw in t for kw in REDOX_KEYWORDS):
            matched.append(t)
    support = len(matched) / max(len(terms), 1)
    seen = set()
    reps = []
    for t in matched:
        if t not in seen:
            reps.append(t)
            seen.add(t)
        if len(reps) >= 5:
            break
    return support, reps

In [42]:
def generate_contextual_embedding_interpretation_prompt(gene_pair_map, context, embedding_base_path="embedding_saved", enrichment_text=None, metabolic_soft_prior: bool = True):
    """Generate a comprehensive prompt for interpreting SLformer embeddings with cross-context analysis and biological mechanism deduction.

    Args:
        gene_pair_map: {"primary": str, "partner": str}
        context: target cancer context label (e.g., "BRCA")
        embedding_base_path: base directory of saved embedding features
        enrichment_text: optional precomputed enrichment summary string; if provided, it will be inserted verbatim
        metabolic_soft_prior: if True, include subtle, evidence-weighted topic cues; metabolism/cofactor topics are only mentioned when supported
    """
    
    # Load essential data components
    _, overlaps, primary_terms, partner_terms = find_shared_goterms(
        gene_pair_map['primary'], gene_pair_map['partner'], return_terms=True
    )
    primary_features, partner_features = load_embedding_features(gene_pair_map, embedding_base_path)

    # Do not compute enrichment here; enrichment_text should be precomputed upstream
    if primary_features is None and partner_features is None:
        return f"Error: No embedding features found for gene pair {gene_pair_map['primary']}-{gene_pair_map['partner']} using {'Geneformer' if USE_GF else 'cross-attention'} embeddings"

    # Analyze embedding patterns with cross-context focus
    analysis = []
    if primary_features is not None and not primary_features.empty:
        analysis.append(f"Primary Gene ({gene_pair_map['primary']}) Contextual Embedding Analysis:")
        analysis.extend(_analyze_embedding_contexts(primary_features, 'primary'))
        
    if partner_features is not None and not partner_features.empty:
        analysis.append(f"\nPartner Gene ({gene_pair_map['partner']}) Contextual Embedding Analysis:")
        analysis.extend(_analyze_embedding_contexts(partner_features, 'partner'))
    
    embedding_type_str = "Geneformer" if USE_GF else "cross-attention"
    available_contexts = []
    if primary_features is not None and not primary_features.empty:
        available_contexts.extend(primary_features['cancer'].tolist())
    if partner_features is not None and not partner_features.empty:
        available_contexts.extend(partner_features['cancer'].tolist())

    # Very soft, evidence-weighted topic cues
    topic_profile_line = ""
    mech_ranking_hint = (
        "Briefly weigh candidate mechanism families (e.g., signaling, DNA repair, metabolic, epigenetic, cell-cycle, cofactor/redox) "
        "by available evidence; discuss the better-supported one(s) first and provide a focused deep-dive for the top family."
    )
    metabolic_note = ""
    redox_note = ""
    guardrail_line = (
        "Avoid over-weighting any single family without convergent support and make uncertainty explicit when evidence is mixed."
    )
    if metabolic_soft_prior:
        try:
            met_support, met_terms = estimate_metabolic_support(enrichment_text or "")
            redox_support, redox_terms = estimate_redox_support(enrichment_text or "")
            # only surface gentle hints when there's some evidence
            met_threshold = 0.15
            redox_threshold = 0.10
            hints = []
            if met_support >= met_threshold:
                reps = (", ".join(met_terms)) if met_terms else ""
                hints.append(
                    "some enrichment terms point toward metabolic processes"
                    + (f" (e.g., {reps})" if reps else "")
                )
                metabolic_note = (
                    "Where metabolic processes appear among supported families, reflect that proportionally to the evidence; treat as hypothesis, not assumption."
                )
            if redox_support >= redox_threshold:
                reps_r = (", ".join(redox_terms)) if redox_terms else ""
                hints.append(
                    "a subset of terms relates to cofactor/redox handling (e.g., NAD(P)H balancing, PPP, malic enzyme, isocitrate dehydrogenase)"
                    + (f" (e.g., {reps_r})" if reps_r else "")
                )
                redox_note = (
                    "If cofactor/redox topics are among the better-supported families, clarify whether the hypothesized effect concerns cofactor production, consumption, or buffering; avoid assuming directionality."
                )
            if hints:
                topic_profile_line = "Preliminary theme signals (from enrichment): " + "; ".join(hints) + ". These are suggestive rather than definitive."
        except Exception:
            pass  # fail closed with no additional cues

    # Neutral, family-specific deep-dive guidance to increase specificity without being directive
    depth_guidance = (
        "When elaborating the top supported family, include 2–3 concrete specifics to increase mechanistic clarity (choose those that fit the evidence):\n"
        "- If metabolic: name the pathway branch or module (e.g., glycolysis, PPP, one-carbon, fatty acid synthesis), a plausible rate-limiting or control step, the main cofactor usage/production (e.g., ATP, NAD(P)H), and subcellular compartment(s) involved (cytosol, mitochondria, peroxisome).\n"
        "- If DNA repair: specify the subpathway (HR, NHEJ, BER, MMR), key complex(es)/enzymes, damage type addressed, and how the gene pair alters repair choice or efficiency.\n"
        "- If signaling: identify the pathway axis (e.g., MAPK, PI3K/AKT/mTOR), the node(s) most likely impacted, and whether changes are upstream/downstream or via feedback.\n"
        "- If epigenetic: state the chromatin modifier class (e.g., HMT, HDAC), relevant histone/DNA marks, and expected impact on transcriptional programs.\n"
        "- If cell-cycle: indicate the checkpoint/phase, cyclin/CDK modules, and the dependency created by the pair.\n"
        "Tie each specific to the target context with 1–2 sentences (e.g., mutations/expression/activity known in this cancer), and ground claims in the provided embeddings/enrichment."
    )
    query_prompt = f'''
You are a computational biologist specialized in mechanistic interpretation of deep learning models for synthetic lethality prediction. 
Your task is to analyze context-aware embeddings from {embedding_type_str} representations, identifying patterns that 
correlate with cancer-specific biological mechanisms. These embeddings represent learned features distinguishing synthetic lethal relationships 
in specific cancer contexts. Your analysis should focus on the differential activation patterns across cancer types to deduce context-specific 
biological mechanisms.

**Gene Pair Investigation:**
Primary Gene: {gene_pair_map['primary']} (GO Terms: {primary_terms})
Partner Gene: {gene_pair_map['partner']} (GO Terms: {partner_terms})
Shared Biological Processes: {overlaps}
{topic_profile_line}

**{embedding_type_str.capitalize()} Embedding Analysis:**
{''.join(analysis)}

**Target Analysis Context:** {context}
**All Available Contexts:** {', '.join(set(available_contexts))}

**Interpretation Framework:**
The embeddings capture context-specific interaction patterns between genes. Each dimension represents a learned feature detector 
sensitive to particular aspects of gene-gene interactions in specific cancer backgrounds. Your analysis should:
1. Identify dimensions showing context-specific activation patterns
2. Compare activation profiles across cancer types
3. Relate numerical differences to known biological mechanisms
4. Formulate hypotheses about how these patterns explain synthetic lethality in the target context
5. Do not directly quote the enrichment results; instead, use them to inform your biological reasoning
6. {mech_ranking_hint}
{metabolic_note}
{redox_note}

**Mechanism depth (to increase specificity):**
{depth_guidance}

**Detailed Instructions:**
1. **Cross-Context Comparison:**
   - Compare activation values for each dimension across all cancer types
   - Identify dimensions with significant differential activation between contexts
   - Note patterns where high activation correlates with high SL scores in specific cancers

2. **Biological Association:**
   - Link activation patterns to GO terms and known cancer biology
   - Consider how context-specific activation might reflect tissue-specific gene functions
   - Evaluate if patterns align with known synthetic lethal relationships in the target cancer
   - State clearly how the genes interact through certain pathways, state those pathways' names, 
   and how they are involved in the context of the target cancer
   - State clearly the main role of each gene in the context of the target cancer
   - When proposing mechanisms, cite which enriched term(s) support each hypothesis and explain the link (e.g., pathway membership, upstream-downstream regulation, or shared complex).
   - {guardrail_line}

3. **Mechanistic Deduction:**
   - Formulate hypotheses about how observed patterns could represent biological mechanisms
   - Consider if activation differences reflect compensatory relationships or pathway dependencies
   - Deduce how the primary and partner genes' interaction changes across contexts
   - Form the deduction based on solid biological evidence, such as known interactions, pathway involvement, or expression patterns

4. **Context Specialization:**
   - Focus on the specified target context while acknowledging patterns from other cancers
   - Explain why the embedding patterns in the target context might be biologically distinct
   - Consider cancer-specific mutations, expression patterns, or pathway activities

5. **Enrichments:**
    - Integrate the provided enrichment results into your mechanistic hypotheses
    - Do not directly quote the enrichment results; instead, use them to inform your biological reasoning

**Output Requirements:**
- Provide a 4-6 sentence analysis tracing embedding patterns to biological mechanisms
- Quantify key observations (e.g., "Dimension A shows 3-fold lower activation in context B vs other contexts")
- Explicitly connect numerical patterns to biological hypotheses
- Maintain focus on the target context while referencing broader patterns`
- Keep the inference logical and grounded in the provided data
- Format: "Embedding Analysis and Context-Specific Mechanism: [Your response...]"

**Analysis Guidelines:**
- Prioritize evidence from the target context but use other contexts for contrast
- When patterns contradict expectations, report the discrepancy without forced reconciliation
- If multiple hypotheses fit the data, present alternatives with supporting evidence
- Use precise biological language grounded in cancer biology and molecular mechanisms
- Avoid declarative statements not directly supported by the embedding evidence
- Make sure to use clear, precise and logical language, do not repeat the stereotyped expressions 

**Constraints:**
- State the full name of the cancer for the first time, followed by abbreviations, (e.g. lung adenocarcinoma (LUAD))
- Do not reference the model's architecture or training process and Do not directly state the SL score
- State limitations if patterns are ambiguous or lack clear biological correlates
- Maintain numerical precision when referencing embedding values
- Use cancer-specific biological knowledge appropriate to the target context
    '''
    return query_prompt

In [43]:
USE_GF = True
print(generate_contextual_embedding_interpretation_prompt(
    gene_pair_map, context, embedding_base_path="embedding_saved", enrichment_text=enrichment_text
))

embedding_saved/geneformer_important/PRKDC-IDH1
Error: No embedding features found for gene pair IDH1-PRKDC using Geneformer embeddings


In [44]:
USE_GF = False
print(generate_contextual_embedding_interpretation_prompt(
    gene_pair_map, context, embedding_base_path="embedding_saved", enrichment_text=enrichment_text
))

embedding_saved/crossemb_important/PRKDC-IDH1
Error: No embedding features found for gene pair IDH1-PRKDC using cross-attention embeddings


In [45]:
# MedCPT and MPNet text similarity utilities
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModel
import torch, torch.nn.functional as F


# Device and model paths
_device = "cuda:2" if torch.cuda.is_available() else "cpu"
_medcpt_path = PATHS["medcpt_model_dir"]
_mpnet_path = PATHS["all-mpnet-base-v2"]
_bge_path = PATHS["bge-large-en-v1.5"]

# Initialize MedCPT via transformers (use [CLS] of last hidden state)
medcpt_tokenizer = AutoTokenizer.from_pretrained(_medcpt_path, local_files_only=True)
medcpt_model = AutoModel.from_pretrained(_medcpt_path, local_files_only=True)
medcpt_model.to(_device)
medcpt_model.eval()

# Initialize MPNet via SentenceTransformers (uses mean pooling by default)
mpnet_model = SentenceTransformer(_mpnet_path, device=_device, local_files_only=True)
bge_model = SentenceTransformer(_bge_path, device=_device, local_files_only=True)




def _medcpt_encode(texts, max_length=64):
    """Encode texts with MedCPT (transformers) using CLS token and L2 normalization."""
    if not texts:
        return None
    enc = medcpt_tokenizer(
        texts,
        truncation=True,
        padding=True,
        return_tensors='pt',
        max_length=max_length,
    )
    enc = {k: v.to(_device) for k, v in enc.items()}
    with torch.no_grad():
        cls = medcpt_model(**enc).last_hidden_state[:, 0, :]
        cls = F.normalize(cls, p=2, dim=1)
    return cls


def medcpt_text_similarity(text_a, text_b, max_length=64):
    """Cosine similarity using MedCPT (CLS pooling via transformers)."""
    if not text_a or not text_b:
        return float('nan')
    embs = _medcpt_encode([text_a, text_b], max_length=max_length)
    if embs is None or embs.shape[0] < 2:
        return float('nan')
    # cosine of L2-normalized vectors equals dot product
    return float(torch.sum(embs[0] * embs[1]).item())


def mpnet_text_similarity(text_a, text_b):
    """Cosine similarity using MPNet (SentenceTransformers)."""
    if not text_a or not text_b:
        return float('nan')
    embs = mpnet_model.encode([text_a, text_b], normalize_embeddings=True)
    return float(util.cos_sim(embs[0], embs[1]))

def bge_text_similarity(text_a, text_b):
    if not text_a or not text_b:
        return float('nan')
    embs = bge_model.encode([text_a, text_b], normalize_embeddings=True)
    return float(util.cos_sim(embs[0], embs[1]))


KeyError: 'medcpt_model_dir'

In [46]:
# Helpers: find raw LLM-output for a gene pair and compute LLM-output vs enrichment similarities
from pathlib import Path
from typing import Optional, Dict


def compute_llm_vs_enrichment_similarities(
    base_dir: str | Path,
    gene_pair: str,
    cancer_code: str,
    enrichment_text: str,
) -> Optional[Dict[str, object]]:
    """Find LLM-output files for the pair and compute medcpt/mpnet/bge similarities vs enrichment_text.

    Returns a dict with keys:
      - 'medcpt','mpnet','bge' -> [cross_score|None, geneformer_score|None]
      - 'txts' -> {'cross': str, 'geneformer': str}
    """
    base_dir = Path(base_dir)
    llm_path_cross = base_dir / 'cross_emb' / f"{gene_pair}_{cancer_code}.txt"
    llm_path_geneformer = base_dir / 'geneformer_emb' / f"{gene_pair}_{cancer_code}.txt"

    print(llm_path_cross)
    print(llm_path_geneformer)
    def _read_text_safe(p: Path) -> str:
        try:
            return p.read_text(encoding='utf-8').strip()
        except Exception:
            return ""

    llm_txt_cross = _read_text_safe(llm_path_cross)
    llm_txt_geneformer = _read_text_safe(llm_path_geneformer)


    sims: Dict[str, list | None] = {'medcpt': [None, None], 'mpnet': [None, None], 'bge': [None, None]}

    # Only compute similarities when a given text exists
    if llm_txt_cross:
        try:
            sims['medcpt'][0] = medcpt_text_similarity(llm_txt_cross, enrichment_text, max_length=256)
        except Exception:
            sims['medcpt'][0] = None
        try:
            sims['mpnet'][0] = mpnet_text_similarity(llm_txt_cross, enrichment_text)
        except Exception:
            sims['mpnet'][0] = None
        try:
            sims['bge'][0] = bge_text_similarity(llm_txt_cross, enrichment_text)
        except Exception:
            sims['bge'][0] = None

    if llm_txt_geneformer:
        try:
            sims['medcpt'][1] = medcpt_text_similarity(llm_txt_geneformer, enrichment_text, max_length=256)
        except Exception:
            sims['medcpt'][1] = None
        try:
            sims['mpnet'][1] = mpnet_text_similarity(llm_txt_geneformer, enrichment_text)
        except Exception:
            sims['mpnet'][1] = None
        try:
            sims['bge'][1] = bge_text_similarity(llm_txt_geneformer, enrichment_text)
        except Exception:
            sims['bge'][1] = None

    return {'medcpt': sims['medcpt'], 'mpnet': sims['mpnet'], 'bge': sims['bge'], 'txts': {'cross': llm_txt_cross, 'geneformer': llm_txt_geneformer}}


def append_llm_scoring_to_report(
    base_dir: str | Path,
    gene_pair: str,
    cancer_code: str,
    enrichment_text: Optional[str],
    report_path: Optional[str | Path] = None,
):
    """Compute LLM-output vs enrichment similarities and append a structured block to the report file.

    The block will include two sections that exactly match the report format used in the repository:
      - Results from cross-attention embedding
      - Results from geneformer embedding

    Each section contains separators, the analysis text (from LLM output file), and a scoring block.
    """
    base_dir = Path(base_dir)

    if report_path is None:
        label = None
        if 'id2cancer_map' in globals():
            label = str(id2cancer_map.get(cancer_code, id2cancer_map.get(cancer_code.upper(), cancer_code))).lower().replace(' ', '')
        if not label:
            label = cancer_code.lower()
        out_name = f"{gene_pair.replace('-', '_')}_{label}.txt"
        report_path = base_dir / 'cross_val_res' / out_name
    report_path = Path(report_path)

    results = compute_llm_vs_enrichment_similarities(base_dir, gene_pair, cancer_code, enrichment_text)

    def _wrap_text(s: str, width: int = 95) -> str:
        if not s:
            return s
        # collapse whitespace to single spaces so wrapping is consistent
        import re
        s_clean = re.sub(r"\s+", " ", s).strip()
        # break into fixed-width chunks
        return '\n'.join([s_clean[i:i+width] for i in range(0, len(s_clean), width)])

    mc = results.get('medcpt', [None, None])
    mp = results.get('mpnet', [None, None])
    bg = results.get('bge', [None, None])
    txts = results.get('txts', {'cross': '', 'geneformer': ''})

    lines = []

    # Cross-attention section
    lines.append('\nResults from cross-attention embedding:\n')
    lines.append('\n' + '=' * 93 + '\n')
    lines.append('\nEmbedding Analysis and Context-Specific Mechanism:\n')
    # Use the LLM-produced analysis if available; otherwise put placeholder
    cross_text = txts.get('cross', '').strip() or '\n<Missing cross-attention analysis output>\n'
    lines.append(_wrap_text(cross_text) + '\n')
    lines.append('\n' + '=' * 93 + '\n')
    lines.append('\nscoring:\n')
    lines.append(f"MedCPT similarity (LLM output vs enrichment): {mc[0]}")
    lines.append(f"MPNet  similarity (LLM output vs enrichment): {mp[0]}")
    lines.append(f"BGE similarity: {bg[0]}\n")

    # Delimiter between sections (match repository file)
    lines.append('\n' + '*' * 93 + '\n' + '*' * 93 + '\n')

    # Geneformer section
    lines.append('\nResults from geneformer embedding:\n')
    lines.append('\n' + '=' * 93 + '\n')
    lines.append('\nEmbedding Analysis and Context-Specific Mechanism:\n')
    gene_text = txts.get('geneformer', '').strip() or '\n<Missing geneformer analysis output>\n'
    lines.append(_wrap_text(gene_text) + '\n')
    lines.append('\n' + '=' * 93 + '\n')
    lines.append('\nscoring:\n')
    lines.append(f"MedCPT similarity (LLM output vs enrichment): {mc[1]}")
    lines.append(f"MPNet  similarity (LLM output vs enrichment): {mp[1]}")
    lines.append(f"BGE similarity: {bg[1]}\n")

    report_text = '\n'.join(lines)

    # Append to report file
    with report_path.open('w', encoding='utf-8') as f:
        f.write(report_text)


    print(f"Appended formatted LLM-output scoring (cross_attn + geneformer) to {report_path}")
    return report_path

# with open(f'data/explanation_groundtruth/{gene_pair[0]}-{gene_pair[1]}/explanation.txt') as f:
#     groundt = f.read()

enr = enrichment_text
append_llm_scoring_to_report('output/LLM_outputs/with_xgb', f'{gene_pair[0]}-{gene_pair[1]}', context.lower(), enr)


output/LLM_outputs/with_xgb/cross_emb/IDH1-PRKDC_glioma.txt
output/LLM_outputs/with_xgb/geneformer_emb/IDH1-PRKDC_glioma.txt
Appended formatted LLM-output scoring (cross_attn + geneformer) to output/LLM_outputs/with_xgb/cross_val_res/IDH1_PRKDC_glioma.txt


PosixPath('output/LLM_outputs/with_xgb/cross_val_res/IDH1_PRKDC_glioma.txt')